In [66]:
#1
import torch # المكتبة الأساسية لتعلم الآلة (PyTorch)، تُستخدم لبناء وتدريب الشبكات العصبية.
import torch.nn as nn # موديل فرعي من PyTorch يحتوي على الأدوات الجاهزة لبناء طبقات الشبكة العصبية (مثل الطبقات الخطية).
from torchvision import models # مكتبة تحتوي على نماذج ذكاء اصطناعي جاهزة ومدربة مسبقاً (مثل ConvNeXt أو ResNet).
import time # مكتبة للتحكم بالوقت، نستخدمها لحساب المدة التي يستغرقها تدريب الموديل في كل دورة.
import gradio as gr # المكتبة المسؤولة عن بناء واجهة المستخدم (الأزرار، صندوق الرفع، والنتائج) بشكل تفاعلي.
import os # مكتبة للتعامل مع نظام التشغيل، مثل قراءة الملفات أو تحديد المسارات (Paths) في المجلدات.
import random # مكتبة لإنتاج أرقام عشوائية، تُستخدم عادةً لخلط البيانات (Shuffling) لضمان حيادية التدريب.
import kagglehub # أداة مخصصة لتحميل البيانات أو الموديلات مباشرة من منصة Kaggle إلى بيئة العمل.
from torch.utils.data import Dataset, DataLoader # أدوات لتنظيم البيانات وتحميلها في دفعات (Batches) أثناء عملية التدريب.
from torchvision import transforms # مكتبة لمعالجة الصور (تغيير الحجم، القص، التحويل لمصفوفات) قبل إدخالها للموديل.
from PIL import Image # المكتبة الأساسية لفتح الصور ومعالجتها وفهم تنسيقاتها المختلفة (مثل PNG و JPEG).

In [67]:
#2

# تحميل الداتا سيت من كاغل باستخدام الأداة التي استوردناها سابقاً وتخزين مسارها في متغير path.
path = kagglehub.dataset_download("muhammadsaoodsarwar/ai-vs-real-192-class-scene-image-dataset")

# تعريف كلاس (Class) مخصص للتعامل مع صور المشروع وتجهيزها للموديل.
class AIvsRealDataset(Dataset):
    def __init__(self, root_dir, transform=None, mode='train', split_ratio=0.8):
        self.transform = transform # تخزين العمليات (مثل تغيير الحجم) التي سنطبقها على الصور.
        self.image_paths = [] # قائمة فارغة سنضع فيها روابط (مسارات) كل الصور.
        self.labels = [] # قائمة فارغة سنضع فيها تصنيف كل صورة (0 للذكاء الاصطناعي، 1 للحقيقي).
        
        # دمج مسار المجلد الرئيسي مع مجلد datasetfull للوصول للبيانات الفعلية.
        data_root = os.path.join(root_dir, 'datasetfull')
           
        print(f"يتم الآن القراءة من المسار: {data_root}")

        # تحديد الفئات: AI سيأخذ الرقم 0، و Real سيأخذ الرقم 1.
        categories = {'AI' : 0,'Real' : 1}
        
        # الدخول لكل مجلد فئة (AI و Real) لقراءة ما بداخله.
        for folder_name, label in categories.items():
            folder_path = os.path.join(data_root, folder_name)
            if not os.path.exists(folder_path): continue # إذا المجلد غير موجود، يتخطاه.
            
            print(f"جاري معالجة فئة: {folder_name}...")
            
            # استخدام os.walk للبحث داخل المجلدات والمجلدات الفرعية عن ملفات الصور.
            for root, dirs, files in os.walk(folder_path):
                # فلترة الملفات للتأكد من أنها صور فقط (بناءً على صيغتها).
                images = [f for f in files if f.lower().endswith(('png', 'jpg', 'jpeg', 'webp'))]
                
                if len(images) > 0: # عدد الصور اكبر من 0
                    images.sort() # ترتيب الصور أبجدياً لضمان الثبات.
                    random.seed(42) # تثبيت العشوائية لضمان الحصول على نفس النتائج في كل مرة.
                    random.shuffle(images) # خلط ترتيب الصور عشوائياً لتحسين جودة التدريب.
                    
                    # حساب نقطة الفصل (80% للتدريب و 20% للاختبار).
                    split_idx = int(len(images) * split_ratio) # هو هيك صار عنده رقم الصورة الي المفروض عندها يوقف ويفصل الجزء الي قبلها للتدريب وهي والي بعدها للفخص
                    
                    # اختيار الصور بناءً على الوضع الحالي (تدريب أم تحقق).
                    if mode == 'train':
                        selected_images = images[:split_idx] # يأخذ الجزء الأول للتدريب.
                    else:
                        selected_images = images[split_idx:] # يأخذ الجزء المتبقي للتحقق.
                    
                    # إضافة المسار الكامل لكل صورة وتصنيفها للقوائم التي أنشأناها.
                    for img in selected_images:
                        self.image_paths.append(os.path.join(root, img))
                        self.labels.append(label)

    # فنكشن يعيد إجمالي عدد الصور الموجودة في الداتا سيت.
    def __len__(self):
        return len(self.image_paths)

    # فنكشن يجلب صورة واحدة وتطبق عليها التحويلات وتعيدها مع التصنيف الخاص بها.
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB') # فتح الصورة وتحويلها لنظام ألوان RGB.
        if self.transform:
            image = self.transform(image) # تطبيق العمليات (مثل تغيير الحجم) إذا كانت موجودة.
        return image, self.labels[idx]

# 2. تعريف العمليات الحسابية على الصور (تغيير الحجم لـ 224، تحويلها لمصفوفة أرقام، وتوحيد الألوان).
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)), #تثبيت حجم الصور
    transforms.ToTensor(), # تحول الصورة من صيغة (PIL Image) إلى صيغة Tensor (وهي مصفوفة رياضية تفهمها مكتبة PyTorch والـ GPU).
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # تقوم بطرح "المتوسط" (mean) والقسمة على "الانحراف المعياري" (std) لكل بكسل في الصورة
    # الهدف جعل بيانات صورك لها نفس "توزيع الألوان" الذي تدرب عليه الموديل الأصلي
])

# إنشاء كائن (Object) لبيانات التدريب وكائن لبيانات التحقق.
train_dataset = AIvsRealDataset(root_dir=path, mode='train', transform=image_transforms)
val_dataset = AIvsRealDataset(root_dir=path, mode='val', transform=image_transforms)

# التحقق من نجاح العثور على صور قبل البدء.
if len(train_dataset) > 0:
    # إنشاء الـ Loaders التي تقوم بتقسيم البيانات لمجموعات صغيرة (32 صورة) وإرسالها للموديل.
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    print("تم تحميل الصور بنجاح")
else:
    print("لا تزال هنالك مشكلة في العثور على الصور")

يتم الآن القراءة من المسار: /kaggle/input/ai-vs-real-192-class-scene-image-dataset/datasetfull
جاري معالجة فئة: AI...
جاري معالجة فئة: Real...
يتم الآن القراءة من المسار: /kaggle/input/ai-vs-real-192-class-scene-image-dataset/datasetfull
جاري معالجة فئة: AI...
جاري معالجة فئة: Real...
تم تحميل الصور بنجاح
يتم الآن القراءة من المسار: /kaggle/input/ai-vs-real-192-class-scene-image-dataset/datasetfull
جاري معالجة فئة: AI...
جاري معالجة فئة: Real...
يتم الآن القراءة من المسار: /kaggle/input/ai-vs-real-192-class-scene-image-dataset/datasetfull
جاري معالجة فئة: AI...
جاري معالجة فئة: Real...
تم تحميل الصور بنجاح


In [64]:
# بما اننا حفظنا المودل سابقا لا داعي لإنشاءه من جديد وتدريبه لذا لا ننفذ هذه الخلية (خلية بناء المودل) ولا الخلية الي بعدها (خلية تدريب المودل)

# ConvNeXt-Tiny: موديل حديث من شركة Meta، يجمع بين قوة الشبكات التقليدية (CNN) وسرعة التقنيات الحديثة (Transformers).
# ImageNet: قاعدة بيانات عالمية ضخمة تحتوي على ملايين الصور، تُستخدم لتدريب الموديل مسبقاً ليعرف الأشكال الأساسية (ألوان، حواف، أجسام).

def build_model():
    # 1. تحميل موديل ConvNeXt-Tiny مع أوزان مدربة مسبقاً على ImageNet
    # الأوزان الجاهزة بتعطي الموديل "خبرة" سابقة في الأشكال والألوان
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    
    # 2. تعديل الطبقة الأخيرة (Classifier)
    # الموديل الأصلي مصمم ليصنف 1000 نوع، إحنا بدنا نوعين بس (Real vs Fake)
    num_ftrs = model.classifier[2].in_features
    
    # استبدال الطبقة الأخيرة بطبقة جديدة تناسب مشروعنا
    # استخدمنا nn.Sequential عشان نضيف Dropout بيمنع الموديل من "البصم" (Overfitting)
    model.classifier[2] = nn.Sequential(
        nn.Linear(num_ftrs, 1),     # مخرج واحد فقط (القيمة القريبة من 1 = Real، والقريبة من 0 = Fake)
        nn.Sigmoid()                # تحويل المخرج لنسبة مئوية بين 0 و 1
    )
    
    return model

# 3. نقل الموديل للـ GPU (كرت الشاشة) لتسريع التدريب 100 ضعف
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model().to(device)

# 4. تعريف "القاضي" و "المصحح"
# BCELoss: هو المقياس اللي بيعرفنا الموديل قديش غلطان في توقعه (صح أم خطأ).
criterion = nn.BCELoss() 

# AdamW: هي خوارزمية تحسين وهو المحرك اللي بيصلح أخطاء الموديل وبيعدل معلوماته عشان يتحسن.
# lr: هي سرعة التعلم، اخترناها بطيئة جداً عشان الموديل يركز في التفاصيل وما يستعجل.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

print("المودل جاهز")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 197MB/s] 


المودل جاهز


In [65]:
# لا ننفذها

# تحديد عدد المرات الكلية التي سيمر فيها الموديل على كامل البيانات (Epochs)
epochs = 10 

print("بدء عملية التدريب...")

# حلقة تكرارية تبدأ عملية التدريب دورة تلو الأخرى
for epoch in range(epochs):
    start_time = time.time() # تسجيل وقت بداية الدورة الحالية
    
    # --- المرحلة الأولى: التدريب (Training Phase) ---
    model.train() # تفعيل وضع التدريب 
    running_loss = 0.0 # عداد لتجميع مقدار الخطأ (Loss) خلال الدورة
    correct_train = 0 # عداد لعدد الإجابات الصحيحة في التدريب
    total_train = 0   # عداد لإجمالي الصور التي تمت معالجتها في التدريب
    
    # حلقة تمر على البيانات المجهزة في دفعات (Batches)
    for images, labels in train_loader:
        # نقل الصور والتصنيفات إلى المعالج (GPU أو CPU) وتعديل أبعادها لتناسب الموديل
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        
        optimizer.zero_grad()           # مسح الحسابات السابقة (Gradients) لبدء حسابات جديدة لهذه الدفعة
        outputs = model(images)         # عملية التمرير الأمامي: الموديل يعطي توقعه للصورة
        loss = criterion(outputs, labels) # حساب الفجوة بين توقع الموديل والحقيقة (مقدار الخطأ)
        loss.backward()                 # عملية الانتشار العكسي: حساب كيف يجب تعديل كل "وزن" لتقليل الخطأ
        optimizer.step()                # التعديل الفعلي للأوزان بناءً على الحسابات السابقة
        
        running_loss += loss.item() # إضافة قيمة الخطأ لهذه الدفعة إلى الإجمالي
        
        # تحويل نواتج الموديل إلى تصنيف (0 أو 1) بناءً على عتبة 0.5
        predicted = (outputs > 0.5).float()
        total_train += labels.size(0) # زيادة إجمالي عدد الصور
        correct_train += (predicted == labels).sum().item() # زيادة عدد الإجابات الصحيحة إذا طابق التوقع الحقيقة
    
    # حساب متوسط الدقة والخطأ لمرحلة التدريب في هذه الدورة
    train_acc = 100 * correct_train / total_train
    avg_train_loss = running_loss / len(train_loader)
    
    # --- المرحلة الثانية: التحقق (Validation Phase) ---
    model.eval() # تفعيل وضع التقييم (إيقاف تحديث الأوزان والـ Dropout لقياس الأداء الحقيقي)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    # إيقاف حساب الـ Gradients لتوفير الذاكرة وسرعة الحساب لأننا لا ندرّب هنا، بل نختبر فقط
    with torch.no_grad(): 
        for images, labels in val_loader:
            # نقل بيانات التحقق إلى المعالج وتجهيزها
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            
            outputs = model(images) # الموديل يتوقع نتائج صور لم يراها في التدريب
            loss = criterion(outputs, labels) # حساب مقدار الخطأ في الاختبار
            val_loss += loss.item()
            
            predicted = (outputs > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    
    # حساب نتائج التحقق النهائية لهذه الدورة
    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)
    
    # حساب الوقت المستغرق في هذه الدورة (بالدقائق والثواني)
    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
    
    # طباعة ملخص النتائج للدورة الحالية
    print(f"Epoch [{epoch+1}/{epochs}] | Time: {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print("-" * 30)

# حفظ "أوزان" الموديل النهائية في ملف لاستخدامه لاحقاً في التنبؤ
torch.save(model, 'ai_detector_convnext.pth')

# طباعة النتيجة النهائية لأهم دورة (وهي الدورة الأخيرة)
print("\n" + "-" * 30)
print(f"Accuracy of the Last Epoch: {val_acc:.2f}%")

بدء عملية التدريب ...
Epoch [1/15] | Time: 2m 34s
Train Loss: 0.2784 | Train Acc: 90.93%
Val Loss: 0.0451 | Val Acc: 99.64%
تم العثور على أفضل موديل وحفظه بدقة: 99.64%
------------------------------
Epoch [2/15] | Time: 2m 21s
Train Loss: 0.0320 | Train Acc: 99.56%
Val Loss: 0.0093 | Val Acc: 99.84%
تم العثور على أفضل موديل وحفظه بدقة: 99.84%
------------------------------
Epoch [3/15] | Time: 2m 21s
Train Loss: 0.0123 | Train Acc: 99.83%
Val Loss: 0.0047 | Val Acc: 99.95%
تم العثور على أفضل موديل وحفظه بدقة: 99.95%
------------------------------
Epoch [4/15] | Time: 2m 21s
Train Loss: 0.0060 | Train Acc: 99.94%
Val Loss: 0.0034 | Val Acc: 99.95%
تم العثور على أفضل موديل وحفظه بدقة: 99.95%
------------------------------
Epoch [5/15] | Time: 2m 21s
Train Loss: 0.0037 | Train Acc: 99.96%
Val Loss: 0.0028 | Val Acc: 99.95%
تم العثور على أفضل موديل وحفظه بدقة: 99.95%
------------------------------
Epoch [6/15] | Time: 2m 21s
Train Loss: 0.0020 | Train Acc: 99.99%
Val Loss: 0.0024 | Val Acc

In [9]:
#3

# 2. تحميل الموديل الذي حفظته
# تأكد أن ملف 'full_ai_detector_model.pt' موجود في الملفات
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load('/kaggle/input/model/full_ai_detector_model.pt', map_location=device, weights_only=False)
model.eval()

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [60]:
#4

# فنكشن التوقع الخاص بـ Gradio
# فنكشن توقع الصورة
def predict_image(input_img):
    # تحويل الصورة بنفس عمليات التحويل التي تمت على الداتا الاصلية
    img = Image.fromarray(input_img).convert('RGB')
    img_t = image_transforms(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(img_t)
        # نستخدم النتيجة الخام مباشرة كما فعلنا في الكود البسيط الناجح
        prob = output.item() 
        
    # نفس المنطق اللي ضبط معك أول مرة
    if prob > 0.5:
        res_label = "Real"
        # تحويل الرقم لنسبة شكلية فقط للعرض
        conf = prob 
    else:
        res_label = "Fake"
        conf = 1 - prob
        
    precision_text = f"## {res_label} : {conf * 100:.2f}%"

    return res_label, precision_text

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Fake vs Real Image Detector")
    gr.Markdown("Upload an image to verify if it was captured by a camera or generated by AI.")
    
    with gr.Row():
        # عمود المدخلات (Input)
        with gr.Column():
            image_input = gr.Image(
                label="Upload Image", 
                type="numpy", 
                sources=["upload", "webcam"],
                elem_classes="upload-box",
                height=400)
            
            submit_btn = gr.Button("Analyze Image 🔍", variant="primary")
        
        # عمود المخرجات (Output)
        with gr.Column():
            label_output = gr.Label(
                num_top_classes=2, 
                label="Analysis Result"
            )
            exact_output = gr.Markdown("")
            
            #إضافة قسم التفاصيل التقنية
            with gr.Accordion("Technical Details", open=True):
                gr.Markdown(
        f"""
- **Model Architecture:** ConvNeXt-Tiny
- **Hardware Accelerator:** {device.type.upper()}
- **Input Resolution:** 224x224 (RGB)
- **Classification Threshold:** 0.5
- **Normalization:** ImageNet Standards
        """,
        elem_classes="details-box"
    )
    
    # ربط الزر بالفنكشن
    submit_btn.click(fn=predict_image, inputs=image_input, outputs=[label_output, exact_output])

# تشغيل التطبيق
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7903
* Running on public URL: https://53ab9b86732dce4963.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",